# Transformers & Self-Attention (Part 1): Why Transformers + Positional Encoding

**Course:** MAI554 – Deep Learning for NLP | **Lecture:** 05 | **Spring 2026** | **Prof. Anis Koubaa**

---

### Learning Objectives

1. **Understand why RNNs fall short** — the bottlenecks that motivated Transformers
2. **Implement and compare three positional encoding strategies:**
   - **APE** — Absolute Positional Encoding (sinusoidal, from *Attention is All You Need*)
   - **RPE** — Relative Positional Encoding (attention bias, from Shaw et al. / T5)
   - **RoPE** — Rotary Positional Embedding (rotation-based, from LLaMA / GPT-NeoX)
3. **See when and why each approach matters** — with side-by-side visualisations

> **Central idea:** Transformers process all tokens in parallel, so they have *no built-in notion of order*. Positional encoding solves this — but **how** you encode position has major consequences for generalisation, long-context, and relative reasoning.

## Part 0: Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt, math, time, warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available()
                       else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

---
## Part 1: Why Transformers?

RNNs hit **three walls**: (1) the entire input is compressed into one hidden vector $h_T$ — long sentences lose early info, (2) $h_t$ depends on $h_{t-1}$ so tokens must be processed sequentially — GPUs sit idle, (3) information decays over 100+ steps even with LSTMs.

**Transformers solve all three** with self-attention: every token attends to every other token in parallel, with O(1) distance.

---
## Part 2: Positional Encoding — Three Strategies

Since Transformers process all tokens **simultaneously**, they have **no inherent notion of order**. Without positional encoding, "The cat sat on the mat" and "The mat sat on the cat" produce **identical** representations.

Three families of solutions have emerged:

| Strategy | Where injected | Encodes | Used in |
|---|---|---|---|
| **APE** (Absolute) | Added to embeddings | "I am at position 5" | Original Transformer, BERT, GPT-2 |
| **RPE** (Relative) | Added to attention scores | "We are 3 tokens apart" | T5, ALiBi, DeBERTa |
| **RoPE** (Rotary) | Rotates Q and K vectors | Both absolute & relative | LLaMA, GPT-NeoX, Mistral |

We will implement each one from scratch, visualise what it does, and understand **why** the field evolved from APE → RPE → RoPE.

### Strategy 1: APE — Absolute Positional Encoding

**Idea:** Give each position a **fixed vector** and **add** it to the token embedding before attention.

$$\text{Input}_i = \text{Embedding}(\text{token}_i) + PE(i)$$

The original Transformer uses **sinusoidal** waves at different frequencies:

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE(pos, 2i\!+\!1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Analogy:** Like an odometer — dim 0 spins fast (ones), dim 10 spins slower (tens), dim 60 barely moves (thousands). The combination is unique per position.

In [ ]:
def sinusoidal_pe(max_len, d_model):
    """Sinusoidal positional encoding (Vaswani et al., 2017).
    
    Each position gets a unique vector of sine/cosine waves.
    Even dims use sin, odd dims use cos, each at a different frequency.
    """
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()          # (max_len, 1)
    div = torch.exp(torch.arange(0, d_model, 2).float()       # frequency decreases
                    * (-math.log(10000.0) / d_model))          # with higher dims
    pe[:, 0::2] = torch.sin(pos * div)   # even dims: sin
    pe[:, 1::2] = torch.cos(pos * div)   # odd dims:  cos
    return pe

# Generate and inspect
pe = sinusoidal_pe(50, 64)
print(f"Shape: {pe.shape} — 50 positions, 64 dimensions")
print(f"Values bounded in [{pe.min():.1f}, {pe.max():.1f}]")

**Visualising the wave patterns.** The heatmap below shows all 64 dimensions across 50 positions. Low dimensions (bottom) oscillate rapidly — they change with every position. High dimensions (top) oscillate slowly — they encode coarse position ranges. The line plot on the right picks a few specific dimensions so you can see the frequency difference clearly.

In [ ]:
# --- APE Visualisation: heatmap + selected wave patterns ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Left: heatmap — each row is a dimension, columns are positions
ax1.imshow(pe.numpy().T, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax1.set_xlabel('Position'); ax1.set_ylabel('Dimension')
ax1.set_title('APE Heatmap — low dims oscillate fast, high dims slow')

# Right: line plot — a few dimensions to see the frequency difference
for dim, c in [(0,'#e74c3c'), (10,'#e67e22'), (30,'#2ecc71'), (62,'#3498db')]:
    ax2.plot(pe[:, dim].numpy(), color=c, lw=1.5,
             label=f'dim {dim} ({"sin" if dim%2==0 else "cos"})')
ax2.set_xlabel('Position'); ax2.set_ylabel('PE value')
ax2.set_title('Each dimension = a wave at different frequency')
ax2.legend(fontsize=8); ax2.set_ylim(-1.3, 1.3)
plt.tight_layout(); plt.show()

**APE in action.** We take the sentence "the cat sat on the mat" and show three heatmaps side by side: the pure token embedding (word identity only), the positional encoding (position only), and their sum (the final input). Notice that the two occurrences of "the" have identical embeddings but different positional encodings — so the final input distinguishes them.

In [ ]:
# --- APE in action: same word at different positions becomes distinguishable ---
sentence = ["the", "cat", "sat", "on", "the", "mat"]
vocab = {w: i for i, w in enumerate(sorted(set(sentence)))}
d_model = 16

emb = nn.Embedding(len(vocab), d_model)
tok_ids = torch.tensor([vocab[w] for w in sentence])
tok_emb = emb(tok_ids)                                # word identity only
pos_enc = sinusoidal_pe(len(sentence), d_model)        # position only
final   = tok_emb + pos_enc                            # both combined

diff = (final[0] - final[4]).abs().mean().item()
print(f"'the'@pos0 vs 'the'@pos4: mean |diff| = {diff:.4f} — same word, different positions")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, data, title in zip(axes,
    [tok_emb.detach(), pos_enc, final.detach()],
    ['Token Embedding (identity)', 'Positional Encoding (position)', 'Final = Emb + PE']):
    im = ax.imshow(data.numpy(), cmap='RdBu_r', aspect='auto')
    ax.set_yticks(range(len(sentence))); ax.set_yticklabels(sentence)
    ax.set_xlabel('Dimension'); ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Position similarity.** A good positional encoding should make nearby positions more similar than distant ones. Below we compute the cosine similarity between all pairs of position vectors. The bright diagonal confirms: positions close together have high similarity, and it fades with distance.

In [ ]:
# --- APE similarity: nearby positions should be more similar ---
pe_64 = sinusoidal_pe(20, 64)
pe_norm = pe_64 / pe_64.norm(dim=1, keepdim=True)
sim = (pe_norm @ pe_norm.T).numpy()

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(sim, cmap='viridis', vmin=-0.5, vmax=1)
ax.set_xlabel('Position'); ax.set_ylabel('Position')
ax.set_title('APE: Cosine Similarity Between Positions')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

print(f"sim(0,1)={sim[0,1]:.3f}  sim(0,5)={sim[0,5]:.3f}  sim(0,19)={sim[0,19]:.3f}")
print("Nearby positions are more similar — a smooth sense of distance")

**APE Limitation:** APE encodes *absolute* position — "I am at position 5." But language often cares about *relative* distance: "the word **two tokens before** me is a verb." APE doesn't encode this directly. Also, APE is added *before* attention, so the position signal mixes with content and can get diluted through layers.

---

### Strategy 2: RPE — Relative Positional Encoding

**Idea:** Instead of "where am I?", encode "**how far apart are we?**" — and inject this into the **attention scores**, not the embeddings.

$$\text{Attention}(i,j) = \frac{q_i \cdot k_j}{\sqrt{d_k}} + \underbrace{b(i-j)}_{\text{relative bias}}$$

The bias $b(i-j)$ depends only on the **distance** between query position $i$ and key position $j$, not on absolute positions.

**Variants:**
- **Shaw et al. (2018):** learnable bias per relative distance, clipped to $[-K, K]$
- **T5 (Raffel et al., 2020):** learnable bias bucketed into logarithmic distance bins
- **ALiBi (Press et al., 2022):** simple linear penalty $b(i-j) = -m \cdot |i-j|$ (no learned params!)

We implement a simple version: a **learnable bias table** indexed by relative distance.

**RPE implementation.** Below we build a learnable relative bias table. For each pair of tokens $(i, j)$, we compute the relative distance $i - j$, clip it to $[-K, K]$, and look up a learned bias. The left plot shows the raw distance matrix (notice the diagonal is 0, and values increase away from it). The right plot shows the bias values that would be added to attention scores.

In [ ]:
def relative_bias(seq_len, max_dist=8):
    """Learnable relative position bias (Shaw et al. style).
    
    Creates a bias table indexed by clipped relative distance.
    Distance is clipped to [-max_dist, max_dist], so the model
    learns one bias per distance bucket.
    """
    positions = torch.arange(seq_len)
    rel_dist = positions.unsqueeze(0) - positions.unsqueeze(1)  # (seq_len, seq_len)
    rel_dist_clipped = rel_dist.clamp(-max_dist, max_dist) + max_dist  # shift to [0, 2K]
    
    # Learnable bias table: one scalar per distance bucket
    bias_table = nn.Parameter(torch.randn(2 * max_dist + 1) * 0.1)
    bias_matrix = bias_table[rel_dist_clipped]  # (seq_len, seq_len)
    return rel_dist, bias_matrix, bias_table

# Demo: 6 tokens, max distance 4
seq_len = 6
rel_dist, bias_matrix, bias_table = relative_bias(seq_len, max_dist=4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: relative distance matrix
im1 = ax1.imshow(rel_dist.numpy(), cmap='RdBu_r', aspect='equal')
ax1.set_title('Relative Distance Matrix (i - j)')
ax1.set_xlabel('Key position j'); ax1.set_ylabel('Query position i')
for i in range(seq_len):
    for j in range(seq_len):
        ax1.text(j, i, f'{rel_dist[i,j].item():+d}', ha='center', va='center', fontsize=9)
plt.colorbar(im1, ax=ax1)

# Right: the bias that gets added to attention scores
im2 = ax2.imshow(bias_matrix.detach().numpy(), cmap='RdBu_r', aspect='equal')
ax2.set_title('RPE Bias Added to Attention Scores')
ax2.set_xlabel('Key position j'); ax2.set_ylabel('Query position i')
plt.colorbar(im2, ax=ax2)
plt.tight_layout(); plt.show()

print("Key insight: RPE adds a learnable bias to attention scores based on DISTANCE,")
print("not absolute position. 'cat' attending to 'the' (distance -1) gets the same")
print("bias regardless of whether it happens at positions 1→0 or 4→3.")

**ALiBi — the simplest RPE.** ALiBi uses no learned parameters at all. It simply subtracts a linear penalty proportional to the distance between tokens: $b(i,j) = -m \cdot |i-j|$. Nearby tokens get near-zero penalty; distant tokens get a large negative bias that softmax pushes toward zero attention weight.

In [ ]:
# --- ALiBi: the simplest RPE — just a linear distance penalty, no learned params ---
def alibi_bias(seq_len, slope=0.1):
    """ALiBi (Press et al., 2022): bias = -slope * |i - j|."""
    pos = torch.arange(seq_len)
    return -slope * (pos.unsqueeze(0) - pos.unsqueeze(1)).abs().float()

alibi = alibi_bias(20, slope=0.1)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(alibi.numpy(), cmap='RdBu_r', aspect='equal')
ax.set_title('ALiBi: Linear Distance Penalty\n(no learned parameters)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

print("ALiBi: nearby tokens get ~0 penalty, distant tokens get large negative bias.")
print("After softmax, this naturally downweights distant tokens — a built-in recency bias.")

---

### Strategy 3: RoPE — Rotary Positional Embedding

**Idea:** Instead of *adding* position to embeddings (APE) or *biasing* attention scores (RPE), **rotate** the Q and K vectors by an angle proportional to their position. The dot product $q_i \cdot k_j$ then *naturally* becomes a function of relative position $(i - j)$.

**The rotation:** For each pair of dimensions $(2i, 2i\!+\!1)$, apply a 2D rotation by angle $\theta_i \cdot pos$:

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(\theta_i \cdot pos) & -\sin(\theta_i \cdot pos) \\ \sin(\theta_i \cdot pos) & \cos(\theta_i \cdot pos) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d}$ (same base frequency as sinusoidal PE).

**Why it works:** When computing $q_i^T k_j$, the rotation angles for positions $i$ and $j$ combine as $\theta \cdot (i - j)$ — so the dot product depends on **relative distance**, even though each vector is rotated by its **absolute position**.

**Best of both worlds:** encodes absolute position (each Q/K is rotated differently) but attention scores depend on relative distance (the rotation difference).

**RoPE implementation.** We apply the 2D rotation to each pair of dimensions in Q and K. Below we take the same vector, place it at 6 different positions, and show that RoPE produces different outputs at each position — even though the input is identical. The rotation angle increases with position.

In [ ]:
def apply_rope(x, seq_len, d_model):
    """Apply Rotary Positional Embedding to a (seq_len, d_model) tensor.
    
    For each pair of dimensions (2i, 2i+1), rotate by angle theta_i * pos.
    This is what LLaMA, Mistral, and GPT-NeoX use.
    """
    # Frequencies: theta_i = 1 / 10000^(2i/d)
    freqs = 1.0 / (10000.0 ** (torch.arange(0, d_model, 2).float() / d_model))
    
    # Angles: theta_i * pos for each position and each frequency
    pos = torch.arange(seq_len).float()
    angles = pos.unsqueeze(1) * freqs.unsqueeze(0)  # (seq_len, d_model//2)
    
    cos_a, sin_a = angles.cos(), angles.sin()
    
    # Split x into pairs of dims and apply 2D rotation
    x1, x2 = x[..., 0::2], x[..., 1::2]             # even and odd dims
    rotated = torch.zeros_like(x)
    rotated[..., 0::2] = x1 * cos_a - x2 * sin_a     # rotation formula
    rotated[..., 1::2] = x1 * sin_a + x2 * cos_a
    return rotated

# Demo: rotate a query vector at different positions
d = 8; seq_len = 6
q = torch.randn(1, d).expand(seq_len, -1)  # same vector at all positions
q_rotated = apply_rope(q, seq_len, d)

print(f"Same vector before RoPE (first 4 dims):")
print(f"  pos 0: [{q[0,:4].tolist()}]")
print(f"  pos 5: [{q[5,:4].tolist()}]")
print(f"\nAfter RoPE rotation:")
print(f"  pos 0: [{q_rotated[0,:4].tolist()}]")
print(f"  pos 5: [{q_rotated[5,:4].tolist()}]")
print("\nSame input, but RoPE rotates each position differently!")

**RoPE's key property: attention becomes position-aware.** Below we compute attention scores $QK^T$ with and without RoPE. Without RoPE, the scores have no positional structure — they are random. With RoPE, a diagonal pattern emerges because the rotation makes nearby tokens interact differently than distant ones.

In [ ]:
# --- RoPE's key property: Q·K depends on RELATIVE position ---
d = 64; N = 20
Q = torch.randn(N, d)
K = torch.randn(N, d)

# Apply RoPE to both Q and K
Q_rope = apply_rope(Q, N, d)
K_rope = apply_rope(K, N, d)

# Compute attention scores with and without RoPE
scores_no_rope = (Q @ K.T) / math.sqrt(d)
scores_rope    = (Q_rope @ K_rope.T) / math.sqrt(d)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = ax1.imshow(scores_no_rope.detach().numpy(), cmap='RdBu_r', aspect='equal')
ax1.set_title('Attention Scores WITHOUT RoPE\n(no position awareness)')
ax1.set_xlabel('Key position'); ax1.set_ylabel('Query position')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(scores_rope.detach().numpy(), cmap='RdBu_r', aspect='equal')
ax2.set_title('Attention Scores WITH RoPE\n(position-aware via rotation)')
ax2.set_xlabel('Key position'); ax2.set_ylabel('Query position')
plt.colorbar(im2, ax=ax2)
plt.tight_layout(); plt.show()

print("Without RoPE: scores are random — no position structure.")
print("With RoPE: diagonal pattern emerges — nearby tokens interact differently than distant ones.")

---

### Side-by-Side: How Each Strategy Affects Attention

All three strategies answer the same question — "how should position influence which tokens attend to each other?" — but they do it at **different points** in the computation:

| | APE | RPE | RoPE |
|---|---|---|---|
| **Where** | Added to input embeddings | Added to attention logits | Applied to Q and K |
| **When** | Before projection to Q/K/V | After $QK^T$ is computed | Before $QK^T$ is computed |
| **What's modified** | The token representation itself | The attention score matrix | The Q and K vectors |
| **Position type** | Absolute only | Relative only | Both (absolute encoding, relative effect) |

**Final comparison.** We apply all three strategies to the same random Q and K matrices and visualise the resulting attention scores side by side. This is the clearest way to see how each approach shapes what the model "sees": APE subtly shifts content, RPE/ALiBi explicitly biases the diagonal, and RoPE bakes position into the geometry of Q and K.

In [ ]:
# --- Side-by-side comparison: how each strategy shapes attention ---
N, d = 20, 64
torch.manual_seed(42)
Q, K = torch.randn(N, d), torch.randn(N, d)
base_scores = (Q @ K.T) / math.sqrt(d)

# 1. APE: add positional encoding to Q and K before computing scores
pe_ape = sinusoidal_pe(N, d)
Q_ape, K_ape = Q + pe_ape, K + pe_ape
scores_ape = (Q_ape @ K_ape.T) / math.sqrt(d)

# 2. RPE (ALiBi): add distance bias to attention scores
scores_rpe = base_scores + alibi_bias(N, slope=0.3)

# 3. RoPE: rotate Q and K, then compute scores
scores_rope = (apply_rope(Q, N, d) @ apply_rope(K, N, d).T) / math.sqrt(d)

# Visualise all three
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, scores, title in zip(axes,
    [base_scores, scores_ape, scores_rpe, scores_rope],
    ['No Position Info', 'APE (add to embeddings)', 'RPE/ALiBi (bias scores)', 'RoPE (rotate Q,K)']):
    im = ax.imshow(scores.detach().numpy(), cmap='RdBu_r', aspect='equal',
                   vmin=-3, vmax=3)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.colorbar(im, ax=axes[-1])
plt.suptitle('How Each PE Strategy Shapes Attention Scores', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print("No PE:   random scores — model has no sense of order")
print("APE:     subtle position signal mixed into content")  
print("RPE:     explicit diagonal bias — nearby tokens are upweighted")
print("RoPE:    position baked into Q·K geometry — smooth relative decay")

---
## Summary & Key Takeaways

| Concept | Key Point |
|---|---|
| **RNN bottlenecks** | Sequential processing, encoding bottleneck, long-range decay |
| **APE** | Fixed vector **added** to embeddings — simple, but only encodes absolute position |
| **RPE** | Bias **added** to attention scores — encodes relative distance directly |
| **RoPE** | **Rotates** Q and K — encodes absolute position but $QK^T$ naturally depends on relative distance |

**The evolution:**
- **APE** (2017): simple and effective, but position signal dilutes through layers and doesn't encode relative distance
- **RPE** (2018-2022): directly encodes what matters (distance), but adds extra parameters or fixed heuristics
- **RoPE** (2021+): elegant mathematical trick — rotation makes $q_i \cdot k_j$ depend on $(i-j)$ without extra parameters. Used in LLaMA, Mistral, GPT-NeoX and most modern LLMs

### What's Next (Part 2)

Now that tokens have both **identity** (embeddings) and **position** (via APE/RPE/RoPE), we're ready for the core mechanism: **Self-Attention** — how every token looks at every other token to build context-aware representations.